In [6]:
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyreadr

In [7]:
resultado = pyreadr.read_r(r"..\dados\comScrubbing\zv_pm_cc400_tc100_SITEdxGroup_spCorr_tcOk.RData")
phenotipo = pyreadr.read_r(r"..\dados\phenotypeComplete.RData")
rois_interesse = pyreadr.read_r(r"..\dados\roisInteresse.RData")
labels_scrubbing = pyreadr.read_r(r"..\dados\labelsComScrubbing_Nclusters_collection\labelsComScrubbing_5clusters.RData")

In [8]:
clusters = labels_scrubbing["labels"].copy()
matriz_conectividade = resultado["dataset"].copy()
grupos = resultado["dxGroup"].copy()
df_phenotipo = phenotipo["phenotype"].copy()

### Construção dos raios espectrais

In [9]:
grupos.value_counts()

dxGroup
1.0        529
2.0        285
3.0         82
Name: count, dtype: int64

In [10]:
clusters.value_counts()

labels
3         92
1         69
2         56
5         54
4         45
Name: count, dtype: int64

In [11]:
index_controle = grupos[grupos["dxGroup"] == 1].index.values
index_autismo = grupos[grupos["dxGroup"] == 2].index.values

matriz_z_score_controle = matriz_conectividade[index_controle, :, :]
matriz_z_score_autismo = matriz_conectividade[index_autismo, :, :]

In [12]:
print(matriz_z_score_controle.shape)
print(matriz_z_score_autismo.shape)

(529, 316, 316)
(285, 316, 316)


In [13]:
def z_score_to_p_valores(z_scores):
    z_absoluto = np.abs(z_scores)
    p_valores = 2 * norm.sf(z_absoluto, loc=0, scale=1)
    return p_valores


def constroi_matrizes_adj(z_scores, alpha=0.05, ajuste="fdr_bh"):
    matriz_p_valor = z_score_to_p_valores(z_scores)
    matriz_completa = []

    for i, matriz_p in enumerate(matriz_p_valor):
        indices_triang_sup = np.triu_indices_from(matriz_p, k=1)
        p_valores_unicos = matriz_p[indices_triang_sup]

        # 2. Aplicar a correção FDR (método Benjamini-Hochberg)
        rejeitados, p_corrigidos_fdr, _, _ = multipletests(p_valores_unicos, alpha=alpha, method=ajuste)

        # 3. Reconstruir a matriz com os p-valores corrigidos
        matriz_p_corrigida = np.zeros_like(matriz_p)
        matriz_p_corrigida[indices_triang_sup] = p_corrigidos_fdr
        matriz_p_corrigida = matriz_p_corrigida + matriz_p_corrigida.T
        np.fill_diagonal(matriz_p_corrigida, 1)

        matriz_completa.append(matriz_p_corrigida)

    matriz_completa = np.array(matriz_completa)
    matriz_adj = (matriz_completa < alpha).astype(int)
    return matriz_adj


def constroi_raios_espectrais(matriz_adj, clusters):
    dict = {}
    for cluste in range(1, 6):
        print(f"Cluster {cluste}: {clusters[clusters['labels'] == cluste].index + 1}")
        index_rois = clusters[clusters["labels"] == cluste].index
        idx_matrizes = np.arange(matriz_adj.shape[0])
        tensor = matriz_adj[np.ix_(idx_matrizes, index_rois, index_rois)]
        dict[f"Cluster_{cluste}"] = []

        for matriz in tensor:
            autovalores = np.linalg.eigvalsh(matriz)
            maior_autovalor = autovalores[-1] 
            
            dict[f"Cluster_{cluste}"].append(maior_autovalor)
    df_autovalores = pd.DataFrame(dict)
    return df_autovalores

In [14]:
matriz_adj_controle = constroi_matrizes_adj(matriz_z_score_controle)
matriz_adj_autismo = constroi_matrizes_adj(matriz_z_score_autismo)

df_controle = constroi_raios_espectrais(matriz_adj_controle, clusters) 
df_autismo = constroi_raios_espectrais(matriz_adj_autismo, clusters)

Cluster 1: Index([  1,  17,  18,  21,  24,  36,  43,  45,  46,  47,  48,  49,  55,  63,
        67,  77,  78,  82,  95,  96, 101, 107, 109, 114, 116, 120, 125, 130,
       132, 135, 138, 144, 153, 159, 160, 161, 162, 168, 172, 175, 182, 183,
       184, 185, 187, 193, 203, 207, 210, 215, 216, 223, 229, 233, 239, 244,
       246, 248, 253, 264, 268, 272, 289, 297, 304, 305, 310, 311, 312],
      dtype='int64')
Cluster 2: Index([  2,  15,  23,  28,  33,  35,  50,  52,  57,  70,  71,  76,  80,  81,
        87,  90, 102, 119, 121, 126, 129, 131, 136, 146, 148, 149, 163, 164,
       167, 173, 178, 180, 189, 192, 196, 201, 209, 214, 224, 225, 234, 236,
       251, 252, 254, 266, 269, 285, 287, 288, 290, 293, 298, 300, 303, 309],
      dtype='int64')
Cluster 3: Index([  3,   4,   6,   7,   9,  10,  11,  12,  14,  16,  19,  22,  26,  31,
        34,  38,  40,  44,  51,  54,  56,  60,  61,  65,  72,  73,  75,  79,
        85,  88,  91,  94,  97, 104, 105, 106, 111, 113, 115, 117, 123, 124,
    

In [15]:
mapping_columns = {"Cluster_1": "Somatomotor",
                   "Cluster_2": "Visual",
                   "Cluster_3": "Default_mode",
                   "Cluster_4": "Cerebellar",
                   "Cluster_5": "Control"}

df_controle.rename(columns=mapping_columns, inplace=True)
df_autismo.rename(columns=mapping_columns, inplace=True)

- Cluster 1: Somatomotor
- Cluster 2: Visual
- Cluster 3: Default-mode
- Cluster 4: Cerebellar
- Cluster 5: Control (Cognitive Control)

In [16]:
df_autismo.to_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo.csv", index=False)
df_controle.to_csv(r"..\dados\raios_espectrais\raios_espectrais_controle.csv", index=False)

In [17]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# def z_score_to_p_valores(z_scores): ...
# def constroi_matrizes_adj(z_scores, alpha=0.05, ajuste="fdr_bh"): ...
# (Mantenha a constroi_matrizes_adj que te mandei anteriormente)

def raios_epectrais_v2(matriz, clusters):
    dict_autovalores = {}
    
    # CORREÇÃO AQUI: Garante que estamos trabalhando com um array numpy puro
    # Se a matriz vier do Xarray ou Pandas, isso extrai apenas os números.
    if hasattr(matriz, 'values'):
        matriz_np = matriz.values
    else:
        matriz_np = np.array(matriz)
        
    # Índices para manter a dimensão dos pacientes
    idx_matrizes = np.arange(matriz_np.shape[0])

    for cluste in range(1, 6):
        print(f"Processando Cluster {cluste}...")
        
        # Garante que seja um array 1D com os índices das ROIs
        index_rois = clusters[clusters["labels"] == cluste].index.values
        
        # Agora o np.ix_ vai funcionar perfeitamente na matriz numpy
        tensor = matriz_np[np.ix_(idx_matrizes, index_rois, index_rois)]
        
        # Passa o tensor inteiro de uma vez
        matrizes_adj = constroi_matrizes_adj(tensor)
        
        # Calcula os autovalores de forma vetorizada
        autovalores = np.linalg.eigvalsh(matrizes_adj)
        
        # O último autovalor é o maior (Raio Espectral)
        maiores_autovalores = autovalores[:, -1] 
        
        dict_autovalores[f"Cluster_{cluste}"] = maiores_autovalores

    df_autovalores = pd.DataFrame(dict_autovalores)
    return df_autovalores
        
# Chamando a função:
df_controlev2 = raios_epectrais_v2(matriz_z_score_controle, clusters)
df_autismov2 = raios_epectrais_v2(matriz_z_score_autismo, clusters)

Processando Cluster 1...
Processando Cluster 2...
Processando Cluster 3...
Processando Cluster 4...
Processando Cluster 5...
Processando Cluster 1...
Processando Cluster 2...
Processando Cluster 3...
Processando Cluster 4...
Processando Cluster 5...


In [18]:
mapping_columns = {"Cluster_1": "Visual",
                   "Cluster_2": "Somatomotor",
                   "Cluster_3": "Default_mode",
                   "Cluster_4": "Cerebellar",
                   "Cluster_5": "Control"}

df_controlev2.rename(columns=mapping_columns, inplace=True)
df_autismov2.rename(columns=mapping_columns, inplace=True)

df_autismov2.to_csv(r"..\dados\raios_espectrais\raios_espectrais_autismov2.csv", index=False)
df_controlev2.to_csv(r"..\dados\raios_espectrais\raios_espectrais_controlev2.csv", index=False)

### Preprocessamento com Rank e Padronização

In [19]:
def rank_padronizacao(df):
    df_preprocessed = df.rank()
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(df_preprocessed)
    df_preprocessed[df_preprocessed.columns] = x_scaled
    return df_preprocessed

In [20]:
df_controle_preprocessado = rank_padronizacao(df_controle)
df_autismo_preprocessado = rank_padronizacao(df_autismo)


df_controle_preprocessadov2 = rank_padronizacao(df_controlev2)
df_autismo_preprocessadov2 = rank_padronizacao(df_autismov2)

In [21]:
df_controle_preprocessado.to_csv(r"..\dados\raios_espectrais\raios_espectrais_controle_prep.csv", index=False)
df_autismo_preprocessado.to_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo_prep.csv", index=False)

In [22]:
df_controle_preprocessadov2.to_csv(r"..\dados\raios_espectrais\raios_espectrais_controle_prepv2.csv", index=False)
df_autismo_preprocessadov2.to_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo_prepv2.csv", index=False)